# EN3160 Assignment 03

## Neural Network

---

**Name:** R.A.C.D. Ranawaka    
**Index Number:** 230526A  
**GitHub Link:** https://github.com/chaniruranawaka/Neural-Network

---

### Q1. Single Dense Layer Network

In [7]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

In [10]:
# Image transformations
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5),
                         (0.5, 0.5, 0.5))
])

# Hyperparameters
batch_size = 50

# Training dataset
trainset = torchvision.datasets.CIFAR10(
    root='./data',
    train=True,
    download=False,
    transform=transform
)

trainloader = torch.utils.data.DataLoader(
    trainset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0
)

# Testing dataset
testset = torchvision.datasets.CIFAR10(
    root='./data',
    train=False,
    download=False,
    transform=transform
)

testloader = torch.utils.data.DataLoader(
    testset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0
)

# CIFAR-10 classes
classes = (
    'plane', 'car', 'bird', 'cat', 'deer',
    'dog', 'frog', 'horse', 'ship', 'truck'
)

print("Training images:", len(trainset))
print("Testing images:", len(testset))

RuntimeError: Dataset not found or corrupted. You can use download=True to download it

In [ ]:
# Network dimensions
Din = 3 * 32 * 32   # Input: 3072
H = 100             # Hidden layer: 100 neurons
K = 10              # Output: 10 classes

# Weight initialization
std = 1e-5

W1 = torch.randn(Din, H) * std
b1 = torch.zeros(H)

W2 = torch.randn(H, K) * std
b2 = torch.zeros(K)

# Training hyperparameters
epochs = 10
lr = 2e-6
lr_decay = 0.9
reg = 0

# History for plotting
loss_history = []
epoch_losses = []

print("W1 shape:", W1.shape)
print("b1 shape:", b1.shape)
print("W2 shape:", W2.shape)
print("b2 shape:", b2.shape)

In [ ]:
for epoch in range(epochs):

    running_loss = 0.0

    for i, data in enumerate(trainloader):

        # Get inputs and labels
        inputs, labels = data

        N = inputs.shape[0]

        # Flatten images: (N, 3, 32, 32) -> (N, 3072)
        X = inputs.view(N, -1)

        # --------------------------------------------------
        # 1. Forward propagation
        # --------------------------------------------------

        # Hidden layer
        Z1 = X.mm(W1) + b1
        A1 = torch.sigmoid(Z1)

        # Output layer (logits)
        Z2 = A1.mm(W2) + b2

        # --------------------------------------------------
        # 2. Cross-entropy loss
        # --------------------------------------------------

        loss = nn.functional.cross_entropy(Z2, labels)

        loss_history.append(loss.item())
        running_loss += loss.item()

        # --------------------------------------------------
        # 3. Backpropagation
        # --------------------------------------------------

        # Gradient of loss with respect to output logits
        P = torch.softmax(Z2, dim=1)

        Y = nn.functional.one_hot(
            labels, num_classes=K
        ).float()

        dZ2 = (P - Y) / N

        # Output-layer gradients
        dW2 = A1.t().mm(dZ2) + reg * W2
        db2 = dZ2.sum(dim=0)

        # Backpropagate through the hidden layer
        dA1 = dZ2.mm(W2.t())

        dZ1 = dA1 * A1 * (1 - A1)

        # Hidden-layer gradients
        dW1 = X.t().mm(dZ1) + reg * W1
        db1 = dZ1.sum(dim=0)

        # --------------------------------------------------
        # 4. Update parameters
        # --------------------------------------------------

        W1 -= lr * dW1
        b1 -= lr * db1

        W2 -= lr * dW2
        b2 -= lr * db2

    # Calculate average loss for the epoch
    epoch_loss = running_loss / len(trainloader)
    epoch_losses.append(epoch_loss)

    print(
        f"Epoch {epoch + 1}/{epochs}, "
        f"Loss: {epoch_loss:.4f}"
    )

    # Learning-rate decay
    lr *= lr_decay

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    range(1, epochs + 1),
    epoch_losses,
    marker='o'
)

plt.title('Training Loss Across 10 Epochs')
plt.xlabel('Epoch')
plt.ylabel('Cross-Entropy Loss')
plt.xticks(range(1, epochs + 1))
plt.grid(True)

plt.show()

In [ ]:
correct_train = 0
total_train = 0

with torch.no_grad():

    for inputs, labels in trainloader:

        N = inputs.shape[0]
        X = inputs.view(N, -1)

        # Forward propagation
        Z1 = X.mm(W1) + b1
        A1 = torch.sigmoid(Z1)

        Z2 = A1.mm(W2) + b2

        # Predicted classes
        predicted = torch.argmax(Z2, dim=1)

        total_train += labels.size(0)

        correct_train += (
            predicted == labels
        ).sum().item()

train_acc = 100 * correct_train / total_train

print(f"Training accuracy: {train_acc:.2f}%")

In [ ]:
correct_test = 0
total_test = 0

with torch.no_grad():

    for inputs, labels in testloader:

        N = inputs.shape[0]
        X = inputs.view(N, -1)

        # Forward propagation
        Z1 = X.mm(W1) + b1
        A1 = torch.sigmoid(Z1)

        Z2 = A1.mm(W2) + b2

        # Predicted classes
        predicted = torch.argmax(Z2, dim=1)

        total_test += labels.size(0)

        correct_test += (
            predicted == labels
        ).sum().item()

test_acc = 100 * correct_test / total_test

print(f"Test accuracy: {test_acc:.2f}%")